In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
import datetime as dt

In [125]:
def load_data(df_name, col_name, agg_type='sum'):
    df = pd.read_csv(df_name)
    df['Date'] = pd.to_datetime(df['Date'])
    df['year'] = df['Date'].dt.year
    df = df.set_index('Date')
    if agg_type == 'sum':
        df_new = df.groupby('year').sum()
    elif agg_type == 'mean':
        df_new = df.groupby('year').mean()
    df_new = df_new.rename(columns={'Value': col_name})
    return df_new

In [127]:
food_waste = pd.read_csv('refed_california.csv')
food_waste = food_waste[food_waste['food_type']== 'Produce']

In [134]:
total_precip = load_data('cali_precip.csv', 'Total Precipitation')
avg_temp = load_data('cali_avgtemp.csv', 'Average Temperature', agg_type='mean')
total_cdd = load_data('cali_cooling.csv', 'Total CDD')
total_hdd = load_data('cali_heating.csv', 'Total HDD')
avg_pdsi = load_data('cali_pdsi.csv', 'Avg PDSI',  agg_type='mean')

/var/folders/v5/dh_hrcj926l3mwkqv25krxsw0000gn/T/ipykernel_51339/2609684062.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date'] = pd.to_datetime(df['Date'])
/var/folders/v5/dh_hrcj926l3mwkqv25krxsw0000gn/T/ipykernel_51339/2609684062.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date'] = pd.to_datetime(df['Date'])
/var/folders/v5/dh_hrcj926l3mwkqv25krxsw0000gn/T/ipykernel_51339/2609684062.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date'] = pd.to_datetime(df['Date'])
/var/folders/v5/dh_hrcj926l3mwkqv25krxsw0000gn/T/ipykernel_51339/2609684062.

In [135]:
climate_df = avg_temp.merge(tot_cdd, on='year').merge(tot_hdd, on='year').merge(total_precip, on='year').merge(avg_pdsi, on='year')

In [136]:
climate_df

,Average Temperature,Total CDD,Total HDD,Total Precipitation,Avg PDSI
year,,,,,
2016,60.133333,1023,2445,25.73,-1.383333
2017,60.341667,1167,2449,28.17,-0.030000
2018,60.125000,1101,2546,18.09,-2.884167
2019,58.408333,896,2950,29.12,0.704167
2020,60.516667,1206,2570,12.07,-2.325833
2021,60.350000,1146,2689,18.93,-5.269167
2022,60.050000,1205,2672,14.10,-4.261667
2023,58.250000,866,3085,27.38,0.239167
2024,60.450000,1192,2717,26.09,-0.947500


In [137]:
spoilage_df = food_waste.merge(climate_df, on='year')

In [138]:
spoilage_df.to_csv('spoilage_all.csv')